### Import environment

In [1]:
# Load environment variables from .env and set JAVA_HOME and SPARK_HOME for Spark
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["JAVA_HOME"] = os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = os.getenv("SPARK_HOME")


### Import necessary libraries

In [2]:
# Import SparkSession and functions alias for dataframe operations
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

### Create session Spark

In [3]:
# Build or get a SparkSession running locally on all available cores
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FootballES")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/20 10:41:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Verify my session spark

In [4]:
# Print the SparkSession object to confirm successful creation
print(spark)

### Analyze Data with Spark

In [11]:
# Read the transformed CSV file into a Spark DataFrame with header parsing
df = spark.read.format("csv").options(
    header="true").load("transformed_data.csv")

In [12]:
# Display the first 5 rows of the DataFrame for quick inspection
df.show(5)

+-----+----------+------------+--------------------+---------+---------+
|Round|      Date|      Team 1|              Team 2|FT Team 1|FT Team 2|
+-----+----------+------------+--------------------+---------+---------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|        0|        0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|        2|        0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|        0|        2|
|    1|2020-09-13|FC Barcelona|            Elche CF|     NULL|     NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|     NULL|     NULL|
+-----+----------+------------+--------------------+---------+---------+
only showing top 5 rows


In [13]:
# Create explicit aliases for team and score columns to standardize names
df = df.selectExpr(
    "*",
    "`Team 1` as `HomeTeam`",
    "`Team 2` as `AwayTeam`",
    "`FT Team 1` as `HomeTeamGoals`",
    "`FT Team 2` as `AwayTeamGoals`"
)

In [14]:
# Show a sample after renaming to verify aliases are correct
df.toPandas().head(5)

,Round,Date,Team 1,Team 2,FT Team 1,FT Team 2,HomeTeam,AwayTeam,HomeTeamGoals,AwayTeamGoals
0,1,2020-09-12,SD Eibar,RC Celta Vigo,0,0,SD Eibar,RC Celta Vigo,0,0
1,1,2020-09-12,Granada CF,Athletic Club Bilbao,2,0,Granada CF,Athletic Club Bilbao,2,0
2,1,2020-09-12,Cádiz CF,CA Osasuna,0,2,Cádiz CF,CA Osasuna,0,2
3,1,2020-09-13,FC Barcelona,Elche CF,None,None,FC Barcelona,Elche CF,None,None
4,1,2020-09-13,Real Madrid,Getafe CF,None,None,Real Madrid,Getafe CF,None,None


In [15]:
# Drop original columns that are no longer needed after aliasing
df = df.drop("FT Team 1", "FT Team 2", "Team 1", "Team 2")

In [16]:
# Quick verify of DataFrame structure and values after dropping columns
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|
+-----+----------+------------+--------------------+-------------+-------------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|
+-----+----------+------------+--------------------+-------------+-------------+
only showing top 5 rows


In [17]:
# Compute match result using numeric comparison of Home vs Away goals
df = df.withColumn("Results",
                   F.when(F.col("HomeTeamGoals") > F.col(
                       "AwayTeamGoals"), "HomeTeamWin")
                   .when(F.col("HomeTeamGoals") < F.col("AwayTeamGoals"), "AwayTeamWin")
                   .otherwise("Draw"))

In [288]:
# Show sample rows including the computed Results column
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+-----------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|       Draw|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
only showing top 5 rows


In [19]:
# Compute season start year from the match Date.
# If month >= 7 (July), season starts that calendar year (e.g. Aug 2013 -> 2013-14),
# otherwise the season started the previous year (e.g. May 2013 -> 2012-13).
df = df.withColumn("Date", F.col("Date").cast("date"))
df_result = df.withColumn("Season",
                          F.when(F.month(F.col("Date")) >= 8,
                                 F.concat(F.year(F.col("Date")), F.lit("/"), (F.year(F.col("Date")) + 1)))
                          .otherwise(
                              F.concat((F.year(F.col("Date")) - 1), F.lit("/"), F.year(F.col("Date"))))
                          )

In [20]:
# Show a few rows to verify Season extraction
df_result.toPandas().head(10)

,Round,Date,HomeTeam,AwayTeam,HomeTeamGoals,AwayTeamGoals,Results,Season
0,1,2020-09-12,SD Eibar,RC Celta Vigo,0,0,Draw,2020/2021
1,1,2020-09-12,Granada CF,Athletic Club Bilbao,2,0,HomeTeamWin,2020/2021
2,1,2020-09-12,Cádiz CF,CA Osasuna,0,2,AwayTeamWin,2020/2021
3,1,2020-09-13,FC Barcelona,Elche CF,None,None,Draw,2020/2021
4,1,2020-09-13,Real Madrid,Getafe CF,None,None,Draw,2020/2021
5,1,2020-09-13,Deportivo Alavés,Real Betis,0,1,AwayTeamWin,2020/2021
6,1,2020-09-13,Real Valladolid CF,Real Sociedad,1,1,Draw,2020/2021
7,1,2020-09-13,Villarreal CF,SD Huesca,1,1,Draw,2020/2021
8,1,2020-09-13,Valencia CF,Levante UD,4,2,HomeTeamWin,2020/2021
9,2,2020-09-19,Villarreal CF,SD Eibar,2,1,HomeTeamWin,2020/2021


In [21]:
# Remove rows where goal values are missing to avoid cast/aggregation errors later
df_result = df_result.dropna(subset=["HomeTeamGoals", "AwayTeamGoals"])

In [22]:
# Verify rows after dropping nulls
df_result.toPandas().head(10)

,Round,Date,HomeTeam,AwayTeam,HomeTeamGoals,AwayTeamGoals,Results,Season
0,1,2020-09-12,SD Eibar,RC Celta Vigo,0,0,Draw,2020/2021
1,1,2020-09-12,Granada CF,Athletic Club Bilbao,2,0,HomeTeamWin,2020/2021
2,1,2020-09-12,Cádiz CF,CA Osasuna,0,2,AwayTeamWin,2020/2021
3,1,2020-09-13,Deportivo Alavés,Real Betis,0,1,AwayTeamWin,2020/2021
4,1,2020-09-13,Real Valladolid CF,Real Sociedad,1,1,Draw,2020/2021
5,1,2020-09-13,Villarreal CF,SD Huesca,1,1,Draw,2020/2021
6,1,2020-09-13,Valencia CF,Levante UD,4,2,HomeTeamWin,2020/2021
7,2,2020-09-19,Villarreal CF,SD Eibar,2,1,HomeTeamWin,2020/2021
8,2,2020-09-19,Getafe CF,CA Osasuna,1,0,HomeTeamWin,2020/2021
9,2,2020-09-19,RC Celta Vigo,Valencia CF,2,1,HomeTeamWin,2020/2021


In [23]:
# Create indicator columns for wins/ties to use in group aggregations later
df_result = df_result.withColumn("HomeTeamWin", F.when(F.col("Results") == "HomeTeamWin", 1).otherwise(0)) \
    .withColumn("AwayTeamWin", F.when(F.col("Results") == "AwayTeamWin", 1).otherwise(0)) \
    .withColumn("GameTie", F.when(F.col("Results") == "Draw", 1).otherwise(0))

In [24]:
# Display a small sample of the DataFrame including new indicator columns
df_result.toPandas().head(10)

,Round,Date,HomeTeam,AwayTeam,HomeTeamGoals,AwayTeamGoals,Results,Season,HomeTeamWin,AwayTeamWin,GameTie
0,1,2020-09-12,SD Eibar,RC Celta Vigo,0,0,Draw,2020/2021,0,0,1
1,1,2020-09-12,Granada CF,Athletic Club Bilbao,2,0,HomeTeamWin,2020/2021,1,0,0
2,1,2020-09-12,Cádiz CF,CA Osasuna,0,2,AwayTeamWin,2020/2021,0,1,0
3,1,2020-09-13,Deportivo Alavés,Real Betis,0,1,AwayTeamWin,2020/2021,0,1,0
4,1,2020-09-13,Real Valladolid CF,Real Sociedad,1,1,Draw,2020/2021,0,0,1
5,1,2020-09-13,Villarreal CF,SD Huesca,1,1,Draw,2020/2021,0,0,1
6,1,2020-09-13,Valencia CF,Levante UD,4,2,HomeTeamWin,2020/2021,1,0,0
7,2,2020-09-19,Villarreal CF,SD Eibar,2,1,HomeTeamWin,2020/2021,1,0,0
8,2,2020-09-19,Getafe CF,CA Osasuna,1,0,HomeTeamWin,2020/2021,1,0,0
9,2,2020-09-19,RC Celta Vigo,Valencia CF,2,1,HomeTeamWin,2020/2021,1,0,0


In [25]:
# Print schema to confirm data types of columns before casting
df_result.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: string (nullable = true)
 |-- AwayTeamGoals: string (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: string (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [26]:
# Types: caster les colonnes en place sans créer de nouvelles colonnes
df_result = df_result.withColumn("HomeTeamGoals", F.col("HomeTeamGoals").cast("int")) \
    .withColumn("AwayTeamGoals", F.col("AwayTeamGoals").cast("int"))

In [27]:
# Show a sample after casting to check types and values
df_result.toPandas().head(5)

,Round,Date,HomeTeam,AwayTeam,HomeTeamGoals,AwayTeamGoals,Results,Season,HomeTeamWin,AwayTeamWin,GameTie
0,1,2020-09-12,SD Eibar,RC Celta Vigo,0,0,Draw,2020/2021,0,0,1
1,1,2020-09-12,Granada CF,Athletic Club Bilbao,2,0,HomeTeamWin,2020/2021,1,0,0
2,1,2020-09-12,Cádiz CF,CA Osasuna,0,2,AwayTeamWin,2020/2021,0,1,0
3,1,2020-09-13,Deportivo Alavés,Real Betis,0,1,AwayTeamWin,2020/2021,0,1,0
4,1,2020-09-13,Real Valladolid CF,Real Sociedad,1,1,Draw,2020/2021,0,0,1


In [28]:
# Print final schema to ensure casting succeeded
df_result.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: integer (nullable = true)
 |-- AwayTeamGoals: integer (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: string (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [29]:
# Drop unnecessary columns 'Date' and 'Round'
df_result = df_result.drop("Date", "Round")

In [30]:
# Verify DataFrame after dropping unneeded columns
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+-----------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|       Draw|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
only showing top 5 rows


In [31]:
# Aggregate statistics for home matches using Spark functions
df_home_matches = df_result.groupBy('Season', 'HomeTeam') \
    .agg(F.sum('HomeTeamWin').alias('TotalHomeWin'),
         F.sum('AwayTeamWin').alias('TotalHomeLoss'),
         F.sum('GameTie').alias('TotalHomeTie'),
         F.sum('HomeTeamGoals').alias('HomeScoredGoals'),
         F.sum('AwayTeamGoals').alias('HomeAgainstGoals')
         ).withColumnRenamed('HomeTeam', 'Team')

In [32]:
# Show aggregated home match statistics for review
df_home_matches.show()

+---------+--------------------+------------+-------------+------------+---------------+----------------+
|   Season|                Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|
+---------+--------------------+------------+-------------+------------+---------------+----------------+
|2016/2017|          Sevilla FC|          14|            1|           4|             39|              16|
|2016/2017|          CD Leganés|           5|            8|           6|             22|              23|
|2015/2016|      Sporting Gijón|           7|            8|           4|             28|              28|
|2012/2013|           Getafe CF|           9|            6|           4|             25|              23|
|2017/2018|          CD Leganés|           9|            6|           4|             19|              19|
|2018/2019|Athletic Club Bilbao|           9|            2|           8|             26|              19|
|2015/2016|       Villarreal CF|          12| 

In [33]:
# Aggregate statistics for away matches similarly and rename AwayTeam to Team for joining
df_away_matches = df_result.groupBy('Season', 'AwayTeam') \
    .agg(F.sum('AwayTeamWin').alias('TotalAwayWin'),
         F.sum('HomeTeamWin').alias('TotalAwayLoss'),
         F.sum('GameTie').alias('TotalAwayTie'),
         F.sum('AwayTeamGoals').alias('AwayScoredGoals'),
         F.sum('HomeTeamGoals').alias('AwayAgainstGoals')
         ).withColumnRenamed('AwayTeam', 'Team')

In [34]:
# Display a few aggregated away match rows
df_away_matches.limit(10).show()

+---------+--------------------+------------+-------------+------------+---------------+----------------+
|   Season|                Team|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|
+---------+--------------------+------------+-------------+------------+---------------+----------------+
|2016/2017|          Sevilla FC|           7|            7|           5|             30|              33|
|2016/2017|          CD Leganés|           3|           11|           5|             14|              32|
|2015/2016|      Sporting Gijón|           3|           11|           5|             12|              34|
|2012/2013|           Getafe CF|           4|           11|           4|             18|              34|
|2017/2018|          CD Leganés|           3|           13|           3|             15|              32|
|2018/2019|Athletic Club Bilbao|           4|            9|           6|             15|              26|
|2015/2016|       Villarreal CF|           6| 

In [35]:
# Join home and away aggregated stats on Season and Team to build team-level stats
df_team_stats = df_home_matches.join(
    df_away_matches, ['Season', 'Team'], 'inner')

In [36]:
df_team_stats.toPandas().head(10)

,Season,Team,TotalHomeWin,TotalHomeLoss,TotalHomeTie,HomeScoredGoals,HomeAgainstGoals,TotalAwayWin,TotalAwayLoss,TotalAwayTie,AwayScoredGoals,AwayAgainstGoals
0,2016/2017,Sevilla FC,14,1,4,39,16,7,7,5,30,33
1,2016/2017,CD Leganés,5,8,6,22,23,3,11,5,14,32
2,2015/2016,Sporting Gijón,7,8,4,28,28,3,11,5,12,34
3,2012/2013,Getafe CF,9,6,4,25,23,4,11,4,18,34
4,2017/2018,CD Leganés,9,6,4,19,19,3,13,3,15,32
5,2018/2019,Athletic Club Bilbao,9,2,8,26,19,4,9,6,15,26
6,2015/2016,Villarreal CF,12,3,4,26,12,6,7,6,18,23
7,2018/2019,Real Sociedad,7,6,6,23,20,6,8,5,22,26
8,2014/2015,Villarreal CF,12,6,1,29,17,4,4,11,19,20
9,2013/2014,Real Betis,5,11,3,19,31,1,14,4,17,47


In [333]:
df_team_stats.toPandas()

,Season,Team,TotalHomeWin,TotalHomeLoss,TotalHomeTie,HomeScoredGoals,HomeAgainstGoals,TotalAwayWin,TotalAwayLoss,TotalAwayTie,AwayScoredGoals,AwayAgainstGoals
0,2016/2017,Sevilla FC,14,1,4,39,16,7,7,5,30,33
1,2016/2017,CD Leganés,5,8,6,22,23,3,11,5,14,32
2,2015/2016,Sporting Gijón,7,8,4,28,28,3,11,5,12,34
3,2012/2013,Getafe CF,9,6,4,25,23,4,11,4,18,34
4,2017/2018,CD Leganés,9,6,4,19,19,3,13,3,15,32
...,...,...,...,...,...,...,...,...,...,...,...,...
175,2019/2020,Granada CF,10,6,3,26,16,6,8,5,26,29
176,2013/2014,Elche CF,6,5,8,13,12,3,11,5,17,38
177,2020/2021,RC Celta Vigo,1,3,0,3,10,0,1,4,3,5
178,2014/2015,Málaga CF,8,5,6,26,20,6,11,2,16,28


In [37]:
# Create columns for total wins, losses, ties, scored goals, and against goals
df_team_stats_totals = df_team_stats.withColumn('TotalWins', F.col('TotalHomeWin') + F.col('TotalAwayWin')) \
    .withColumn('TotalLosses', F.col('TotalHomeLoss') + F.col('TotalAwayLoss')) \
    .withColumn('TotalTies', F.col('TotalHomeTie') + F.col('TotalAwayTie')) \
    .withColumn('TotalScoredGoals', F.col('HomeScoredGoals') + F.col('AwayScoredGoals')) \
    .withColumn('TotalAgainstGoals', F.col('HomeAgainstGoals') + F.col('AwayAgainstGoals'))

In [38]:
df_team_stats_totals.toPandas().head(10)

,Season,Team,TotalHomeWin,TotalHomeLoss,TotalHomeTie,HomeScoredGoals,HomeAgainstGoals,TotalAwayWin,TotalAwayLoss,TotalAwayTie,AwayScoredGoals,AwayAgainstGoals,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals
0,2016/2017,Sevilla FC,14,1,4,39,16,7,7,5,30,33,21,8,9,69,49
1,2016/2017,CD Leganés,5,8,6,22,23,3,11,5,14,32,8,19,11,36,55
2,2015/2016,Sporting Gijón,7,8,4,28,28,3,11,5,12,34,10,19,9,40,62
3,2012/2013,Getafe CF,9,6,4,25,23,4,11,4,18,34,13,17,8,43,57
4,2017/2018,CD Leganés,9,6,4,19,19,3,13,3,15,32,12,19,7,34,51
5,2018/2019,Athletic Club Bilbao,9,2,8,26,19,4,9,6,15,26,13,11,14,41,45
6,2015/2016,Villarreal CF,12,3,4,26,12,6,7,6,18,23,18,10,10,44,35
7,2018/2019,Real Sociedad,7,6,6,23,20,6,8,5,22,26,13,14,11,45,46
8,2014/2015,Villarreal CF,12,6,1,29,17,4,4,11,19,20,16,10,12,48,37
9,2013/2014,Real Betis,5,11,3,19,31,1,14,4,17,47,6,25,7,36,78


In [39]:
# Drop unecessary columns
cols_to_drop = ['TotalHomeWin', 'TotalAwayWin', 'TotalHomeLoss', 'TotalAwayLoss',
                'TotalHomeTie', 'TotalAwayTie', 'HomeScoredGoals', 'AwayScoredGoals',
                'HomeAgainstGoals', 'AwayAgainstGoals']
df_final_stats = df_team_stats_totals.drop(*cols_to_drop)

In [40]:
df_final_stats.show()

+---------+--------------------+---------+-----------+---------+----------------+-----------------+
|   Season|                Team|TotalWins|TotalLosses|TotalTies|TotalScoredGoals|TotalAgainstGoals|
+---------+--------------------+---------+-----------+---------+----------------+-----------------+
|2016/2017|          Sevilla FC|       21|          8|        9|              69|               49|
|2016/2017|          CD Leganés|        8|         19|       11|              36|               55|
|2015/2016|      Sporting Gijón|       10|         19|        9|              40|               62|
|2012/2013|           Getafe CF|       13|         17|        8|              43|               57|
|2017/2018|          CD Leganés|       12|         19|        7|              34|               51|
|2018/2019|Athletic Club Bilbao|       13|         11|       14|              41|               45|
|2015/2016|       Villarreal CF|       18|         10|       10|              44|               35|


In [41]:
# Create additional columns for goal difference and points
df_processed = df_final_stats.withColumn('GoalDifference', F.col('TotalScoredGoals') - F.col('TotalAgainstGoals')) \
    .withColumn('Points', F.col('TotalWins') * 3 + F.col('TotalTies') * 1) \
    .withColumn('WinPercentage', F.round(F.col('TotalWins') / (F.col('TotalWins') + F.col('TotalLosses') + F.col('TotalTies')) * 100, 2))

In [42]:
# Preview the final processed DataFrame
df_processed.toPandas()

,Season,Team,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals,GoalDifference,Points,WinPercentage
0,2016/2017,Sevilla FC,21,8,9,69,49,20,72,55.26
1,2016/2017,CD Leganés,8,19,11,36,55,-19,35,21.05
2,2015/2016,Sporting Gijón,10,19,9,40,62,-22,39,26.32
3,2012/2013,Getafe CF,13,17,8,43,57,-14,47,34.21
4,2017/2018,CD Leganés,12,19,7,34,51,-17,43,31.58
...,...,...,...,...,...,...,...,...,...,...
175,2019/2020,Granada CF,16,14,8,52,45,7,56,42.11
176,2013/2014,Elche CF,9,16,13,30,50,-20,40,23.68
177,2020/2021,RC Celta Vigo,1,4,4,6,15,-9,7,11.11
178,2014/2015,Málaga CF,14,16,8,42,48,-6,50,36.84


In [43]:
# drop rows where Season == 2020/2021 because incomplete season
df_processed_final = df_processed.filter(F.col("Season") != "2020/2021")

In [44]:
df_processed_final.toPandas()

,Season,Team,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals,GoalDifference,Points,WinPercentage
0,2016/2017,Sevilla FC,21,8,9,69,49,20,72,55.26
1,2016/2017,CD Leganés,8,19,11,36,55,-19,35,21.05
2,2015/2016,Sporting Gijón,10,19,9,40,62,-22,39,26.32
3,2012/2013,Getafe CF,13,17,8,43,57,-14,47,34.21
4,2017/2018,CD Leganés,12,19,7,34,51,-17,43,31.58
...,...,...,...,...,...,...,...,...,...,...
155,2018/2019,RC Celta Vigo,10,17,11,53,62,-9,41,26.32
156,2019/2020,Granada CF,16,14,8,52,45,7,56,42.11
157,2013/2014,Elche CF,9,16,13,30,50,-20,40,23.68
158,2014/2015,Málaga CF,14,16,8,42,48,-6,50,36.84


In [45]:
# set window partitioning and ordering for ranking teams within each season
window_spec = Window.partitionBy('Season').orderBy(F.col('WinPercentage').desc(
), F.col('GoalDifference').desc(), F.col('TotalScoredGoals').desc())

# Rank Teams by season
df_ranked = df_processed_final.withColumn('Rank', F.rank().over(window_spec))

In [46]:
df_ranked.toPandas()

,Season,Team,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals,GoalDifference,Points,WinPercentage,Rank
0,2012/2013,FC Barcelona,32,2,4,115,40,75,100,84.21,1
1,2012/2013,Real Madrid,26,5,7,103,42,61,85,68.42,2
2,2012/2013,Atlético Madrid,23,8,7,65,31,34,76,60.53,3
3,2012/2013,Valencia CF,19,11,8,67,54,13,65,50.00,4
4,2012/2013,Real Sociedad,18,8,12,70,49,21,66,47.37,5
...,...,...,...,...,...,...,...,...,...,...,...
155,2019/2020,Real Valladolid CF,9,14,15,32,43,-11,42,23.68,16
156,2019/2020,RCD Mallorca,9,23,6,40,65,-25,33,23.68,17
157,2019/2020,CD Leganés,8,18,12,30,51,-21,36,21.05,18
158,2019/2020,RC Celta Vigo,7,15,16,37,49,-12,37,18.42,19


In [47]:
# filter top teams per season
df_top_teams = df_ranked.filter(F.col("Rank") == 1)

# preview data
df_top_teams.toPandas()

,Season,Team,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals,GoalDifference,Points,WinPercentage,Rank
0,2012/2013,FC Barcelona,32,2,4,115,40,75,100,84.21,1
1,2013/2014,Atlético Madrid,28,4,6,77,26,51,90,73.68,1
2,2014/2015,FC Barcelona,30,4,4,110,21,89,94,78.95,1
3,2015/2016,FC Barcelona,29,5,4,112,29,83,91,76.32,1
4,2016/2017,Real Madrid,29,3,6,106,41,65,93,76.32,1
5,2017/2018,FC Barcelona,28,1,9,99,29,70,93,73.68,1
6,2018/2019,FC Barcelona,26,3,9,90,36,54,87,68.42,1
7,2019/2020,Real Madrid,26,3,9,70,25,45,87,68.42,1


In [50]:
# number of titles per team
df_team_titles = df_top_teams.groupBy('Team') \
    .agg(F.count('Season').alias('NumberOfTitles')) \
    .orderBy(F.col('NumberOfTitles').desc())

In [51]:
df_team_titles.toPandas()

,Team,NumberOfTitles
0,FC Barcelona,5
1,Real Madrid,2
2,Atlético Madrid,1


## Save results to CSV

In [58]:
df_ranked.coalesce(1)\
    .write.format("csv")\
    .option("header", "true")\
    .mode("overwrite")\
    .save("results_team_stats")

In [59]:
df_top_teams.coalesce(1)\
    .write.format("csv")\
    .option("header", "true")\
    .mode("overwrite")\
    .save("results_top_teams")

In [60]:
df_team_titles.coalesce(1)\
    .write.format("csv")\
    .option("header", "true")\
    .mode("overwrite")\
    .save("results_team_titles")

## Close the Spark Session

In [357]:
# Close the Spark session
spark.stop()